# 02 — Evidence Assembly

Combine the standardized SEC, market, and news outputs into a **ticker/date-aligned evidence table** for downstream agents.

This notebook organizes evidence only; it does **not** make investment recommendations.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evidence import assemble_daily_evidence  # noqa: E402
from shared import PATHS  # noqa: E402


## 1. Load Standardized Data

In [2]:
sec = pd.read_csv(PATHS["processed"] / "sec_form4_clean.csv")
market = pd.read_csv(PATHS["processed"] / "market_data_clean.csv")
news = pd.read_csv(PATHS["processed"] / "news_clean.csv")

print("SEC rows:", len(sec))
print("Market rows:", len(market))
print("News rows:", len(news))

SEC rows: 57
Market rows: 100
News rows: 24


## 2. Assemble Daily Evidence

In [3]:
evidence = assemble_daily_evidence(
    market=market,
    news=news,
    sec=sec,
)

print("Evidence rows:", len(evidence))
display(evidence.head(20))

Evidence rows: 108


,ticker,date,open,high,low,close,volume,daily_return,volume_change,news_count,news_titles,news_urls,insider_transaction_count,insider_total_value,insider_names
0,AAPL,2026-02-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,1.0,0.00,WAGNER SUSAN
1,AAPL,2026-03-15 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,2.0,8135903.36,Newstead Jennifer
2,AAPL,2026-04-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,15.0,34320628.17,O'BRIEN DEIRDRE || Khan Sabih || COOK TIMOTHY D
3,AAPL,2026-04-02 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,8.0,24173072.77,O'BRIEN DEIRDRE || COOK TIMOTHY D
4,AAPL,2026-04-15 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,10.0,1514654.55,Borders Ben || Parekh Kevan
5,AAPL,2026-04-23 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,1.0,421850.00,Parekh Kevan
6,AAPL,2026-04-28 00:00:00+00:00,272.335,273.23,268.6600,270.71,40018940.0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
7,AAPL,2026-04-29 00:00:00+00:00,267.550,271.04,267.0400,270.17,30047869.0,-0.001995,-0.249159,0,NaN,NaN,NaN,NaN,NaN
8,AAPL,2026-04-30 00:00:00+00:00,270.500,276.00,268.1400,271.35,91848230.0,0.004368,2.056730,0,NaN,NaN,NaN,NaN,NaN
9,AAPL,2026-05-01 00:00:00+00:00,278.855,287.22,278.3700,280.14,79915442.0,0.032394,-0.129919,0,NaN,NaN,NaN,NaN,NaN


## 3. Validate and Save

In [4]:
print("Total evidence rows:", len(evidence))
print("Minimum date:", evidence["date"].min())
print("Maximum date:", evidence["date"].max())
print("Rows containing market data:", evidence["close"].notna().sum())
print(
    "Rows with SEC activity:",
    evidence["insider_transaction_count"].notna().sum(),
)
print("Rows with news:", (evidence["news_count"] > 0).sum())
print("news_count contains no NaN values:", evidence["news_count"].notna().all())
print("Duplicate ticker/date rows:", evidence.duplicated(["ticker", "date"]).sum())

output_path = PATHS["processed"] / "daily_evidence.csv"
evidence.to_csv(output_path, index=False)
print("Saved:", output_path)

Total evidence rows: 108
Minimum date: 2026-02-24 00:00:00+00:00
Maximum date: 2026-09-20 00:00:00+00:00
Rows containing market data: 100
Rows with SEC activity: 17
Rows with news: 4
news_count contains no NaN values: True
Duplicate ticker/date rows: 0
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\daily_evidence.csv


## Handoff to Agent Workflows

The resulting `daily_evidence.csv` is the common evidence layer for:

- prompt chaining;
- routing to specialist agents;
- evaluator–optimizer review.

Downstream agents should cite/source the underlying evidence rather than treating generated text as primary evidence.